# PVT v2 + MoE — MegaBlocks dMoE backend (twin of notebook 01)

Identical to `01_train_supervised.ipynb` except: the install cell below and
`model.moe.backend = "megablocks"`. Everything else (model, data, optimizer,
diagnostics) is the same package code.

**dMoE is dropless** — differences from Tutel you should expect:
- `capacity_factor` and `gate_noise` are **no-ops** (documented, kept for config
  portability). Router jitter, if wanted, is megablocks' `moe_jitter_eps` (not exposed).
- No token dropping means no capacity-overflow behavior to tune.
- No Tutel gate train-forcing is needed (and none runs for this backend).
- The load-balancing loss is collected per-layer in training mode only.

**Why the previous (archived) MegaBlocks attempt failed** — three independent causes,
all fixed in `pvt_moe/models/ffn.py`:
1. `grouped_gemm` was never installed/built -> `mlp_impl='grouped'` asserts at forward.
2. The aux loss was collected in eval mode too -> crash on Lightning's val sanity check
   (the registry is only populated when `module.training`).
3. `bias=True` was passed but grouped MLPs silently create no biases -> the model lost
   all FFN biases and HF bias seeding matched nothing. We set `bias=False` honestly.

In [ ]:
# Environment check — plain Jupyter on B200 (sm_100) or RTX 5090 (sm_120).
# Credentials: export HF_TOKEN / WANDB_API_KEY in the shell that starts Jupyter.
import importlib
import subprocess
import sys

import torch

print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})")
else:
    print("WARNING: no GPU visible — training will not be practical.")

_torch_minor = tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
if _torch_minor < (2, 5):
    print("NOTE: torch >= 2.5 recommended (fast SDPA GQA path; >=2.4 for fused RMSNorm). "
          "The code falls back gracefully but slower.")

_required = ["pytorch_lightning", "torchmetrics", "timm", "datasets", "transformers",
             "huggingface_hub", "pandas", "matplotlib", "fvcore", "wandb"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    print(f"Installing: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [ ]:
# MegaBlocks dMoE backend — read this cell before running it.
#
# Requirements (verified against megablocks 0.10.0, May 2025):
#   * megablocks==0.10.0 pins torch>=2.7.0,<2.7.1 — check your torch first.
#     If your torch differs, install megablocks from source instead (last
#     resort: pip install --no-deps megablocks==0.10.0 and verify manually).
#   * mlp_impl='grouped' (the only viable impl: 'sparse' was disabled upstream
#     in v0.8.0) REQUIRES the grouped_gemm CUTLASS extension — there is NO
#     torch-native fallback.
#   * grouped_gemm compiles CUDA at install time: nvcc + CUDA headers must be
#     present. TORCH_CUDA_ARCH_LIST below covers H100 (9.0), B200 (10.0) and
#     RTX 5090 (12.0). sm_120 support in grouped_gemm 0.3.0 is UNVERIFIED —
#     if the build or the first forward fails on the 5090, use the Tutel
#     notebook (01) on that box and run MegaBlocks on the B200.
import importlib
import os
import subprocess
import sys

import torch

if not torch.__version__.startswith("2.7"):
    print(f"WARNING: torch {torch.__version__} != 2.7.x (megablocks 0.10.0 pin).\n"
          "Options: (a) conda/pip env with torch 2.7.x, or (b) install megablocks "
          "from source against your torch:  pip install --no-build-isolation "
          "git+https://github.com/databricks/megablocks.git")

os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0;10.0;12.0"

if importlib.util.find_spec("grouped_gemm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--no-build-isolation", "grouped_gemm==0.3.0"])
if importlib.util.find_spec("megablocks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "megablocks==0.10.0"])

from megablocks import grouped_gemm_util

assert grouped_gemm_util.grouped_gemm_is_available() if hasattr(grouped_gemm_util, "grouped_gemm_is_available") else grouped_gemm_util.backend is not None, (
    "grouped_gemm did not build — mlp_impl='grouped' cannot run. "
    "Check nvcc/CUDA headers and TORCH_CUDA_ARCH_LIST, or fall back to the Tutel notebook."
)
print("megablocks + grouped_gemm OK")

In [ ]:
# Make the repo importable (notebooks/ lives one level under the repo root).
import pathlib
import sys

cwd = pathlib.Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe package not found under {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import LitClassifier, build_ssl_trainer, build_trainer, setup_environment
from pvt_moe.models import build_model
from pvt_moe.utils import (
    count_flops,
    count_params,
    expert_utilization,
    plot_expert_utilization,
    plot_training_curves,
)

print(f"pvt_moe loaded from {REPO_ROOT}")

In [ ]:
# ============================== CONFIG =====================================
# Same schema as notebook 01; backend switched to megablocks.
cfg = merge_config(default_config(), {
    "mode": "hf_pretrained",
    "ckpt_path": None,
    "epochs": 100,
    "batch_size": 1024,
    "num_workers": 12,
    "use_wandb": True,

    "model": {
        "norm_type": "layernorm",
        "ablation": {
            "use_moe": True,
            "moe_placement": [[], [], [], [0, 1]],
            "use_rope": True,
            "rope_placement": [[], [], [], [0, 1]],
            "rope_theta": None,   # None = per-mode default (mixed 10 / axial 50)
        },
        # capacity_factor / gate_noise are no-ops for megablocks (dropless).
        "moe": {"backend": "megablocks", "num_experts": 8, "top_k": 1},
    },
    "dataset": {"name": "imagenet-1k"},
    "optim": {"lr": 1e-4, "warmup_epochs": 7, "stage4_lr_multiplier": 10.0},
})

cfg = validate_config(cfg)
print("run:", cfg["run_name"])

In [ ]:
device = setup_environment(cfg)

In [ ]:
train_loader, val_loader = build_dataloaders(cfg)
xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape, xb.dtype)

In [ ]:
# MegaBlocks-specific sanity: dMoE constructs, expert layout is recognized,
# aux flows in train mode and is silent in eval (the archived attempt crashed
# exactly here). GPU required for grouped_gemm.
import torch

assert torch.cuda.is_available(), "megablocks grouped path needs CUDA"
_probe = validate_config(merge_config(cfg, {"mode": "scratch",
                                            "model": {"pretrained_hf_id": None}}))
_full = build_model(_probe).cuda()

# dMoE weights are bf16 (fp16 default overridden in ffn.py) and the grouped
# kernels never run fp32 -> forwards must happen under bf16 autocast, exactly
# as they do in bf16-mixed training.
_full.train()
with torch.autocast("cuda", dtype=torch.bfloat16):
    _logits, _aux = _full(torch.randn(2, 3, 224, 224, device="cuda"))
assert _aux is not None and torch.isfinite(_aux), "train-mode aux must flow"
print(f"train aux OK: {_aux.item():.4f}")

_full.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    _logits, _aux_eval = _full(torch.randn(2, 3, 224, 224, device="cuda"))
assert float(_aux_eval) == 0.0, "eval-mode aux must be zero (registry not populated)"
print("eval aux OK (0.0)")

# Print the actual expert parameter layout once — seeding depends on it.
for _n, _m in _full.named_modules():
    if _n.endswith("mlp") and hasattr(_m, "moe_layer"):
        print(f"{_n}: " + str([(k, tuple(v.shape)) for k, v in _m.moe_layer.named_parameters()][:6]))
        break
del _full
torch.cuda.empty_cache()

In [ ]:
# Sanity gauntlet — run before EVERY training launch (seconds of insurance
# against hours of wasted GPU time).
import torch

# 1) Tiny CPU model, all ablations off: forward shape + no-aux contract.
_tiny = validate_config(merge_config(cfg, {
    "mode": "scratch",
    "model": {
        "embed_dims": [16, 32, 48, 64], "num_heads": [1, 2, 4, 4],
        "num_kv_heads": [1, 1, 2, 2], "mlp_ratios": [2, 2, 2, 2],
        "depths": [1, 1, 1, 1], "pretrained_hf_id": None,
        "ablation": {"use_moe": False, "moe_placement": [[], [], [], []],
                     "use_rope": False, "rope_placement": [[], [], [], []]},
    },
}))
_m = build_model(_tiny)
_logits, _aux = _m(torch.randn(2, 3, 224, 224))
assert _logits.shape == (2, cfg["dataset"]["num_classes"]), _logits.shape
assert _aux is None
print("tiny CPU forward OK")

# 2) Full architecture on GPU with the REAL ablation flags: aux must flow
#    through MoE blocks in train mode.
if torch.cuda.is_available():
    _probe = validate_config(merge_config(cfg, {"mode": "scratch",
                                                "model": {"pretrained_hf_id": None}}))
    _full = build_model(_probe).cuda().train()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _logits, _aux = _full(torch.randn(2, 3, 224, 224, device="cuda"))
    assert _logits.shape == (2, cfg["dataset"]["num_classes"])
    _abl = _probe["model"]["ablation"]
    if _abl["use_moe"] and any(_abl["moe_placement"]):
        assert _aux is not None and torch.isfinite(_aux), "MoE on but aux did not flow!"
        print(f"GPU aux-flow OK (aux={_aux.item():.4f})")
    _full.eval()
    del _full
    torch.cuda.empty_cache()
print("sanity checks passed")

In [ ]:
model = LitClassifier(cfg)
count_params(model.model)

In [ ]:
trainer = build_trainer(cfg)
resume_ckpt = cfg["ckpt_path"] if cfg["mode"] == "resume" else None
trainer.fit(model, train_loader, val_loader, ckpt_path=resume_ckpt)

In [ ]:
print("best:", trainer.checkpoint_callback.best_model_path)
print("last:", os.path.join(trainer.checkpoint_callback.dirpath, "last.ckpt"))

In [ ]:
import os

csv_dir = trainer.loggers[0].log_dir  # CSVLogger is always loggers[0]
plot_training_curves(os.path.join(csv_dir, "metrics.csv"),
                     save_path=os.path.join(csv_dir, "training_curves.png"))

In [ ]:
# Expert utilization — THE first diagnostic when an MoE run underperforms.
# Healthy top-1/8-expert routing: every expert 5-25%, entropy near 2.08.
if cfg["model"]["ablation"]["use_moe"]:
    # PL teardown moves the module to CPU after fit — put it back first.
    counts = expert_utilization(model.model.to(device), val_loader, num_batches=20)
    plot_expert_utilization(counts)

In [ ]:
count_flops(model.model, img_size=cfg["dataset"]["img_size"])

## Notes

- **torch pin**: megablocks 0.10.0 requires torch 2.7.x. Newer torch -> install
  megablocks from source (see install cell).
- **RTX 5090 (sm_120)**: grouped_gemm 0.3.0 sm_120 support is unverified. If the
  install-cell assert or the sanity cell fails on the 5090, run this notebook on
  the B200 and use notebook 01 (Tutel) on the 5090.
- **bf16**: dMoE weights are created in bf16 (`Arguments.fp16` defaults True
  upstream — overridden in `pvt_moe/models/ffn.py`) and every MoE forward must run
  under bf16 autocast; Lightning's bf16-mixed does this during training, and the
  sanity/diagnostic cells wrap their own forwards.
- **device**: `Arguments.device` upstream defaults to `torch.cuda.current_device()`
  *at construction* — overridden to CPU in ffn.py so models build like every other
  module and move with `.to(device)`.